In [ ]:
import gymnasium
import numpy as np
from gymnasium.spaces import Discrete, MultiDiscrete, Box, Dict
from gymnasium.utils import seeding

from pettingzoo import AECEnv
from pettingzoo.utils import AgentSelector, wrappers

In [ ]:
def env(render_mode=None):
    """
    The env function often wraps the environment in wrappers by default.
    You can find full documentation for these methods
    elsewhere in the developer documentation.
    """
    internal_render_mode = render_mode if render_mode != "ansi" else "human"
    env = raw_env(render_mode=internal_render_mode)
    # This wrapper is only for environments which print results to the terminal
    if render_mode == "ansi":
        env = wrappers.CaptureStdoutWrapper(env)
    # this wrapper helps error handling for discrete action spaces
    env = wrappers.AssertOutOfBoundsWrapper(env)
    # Provides a wide vareity of helpful user errors
    # Strongly recommended
    env = wrappers.OrderEnforcingWrapper(env)
    return env


class raw_env(AECEnv):
    """
    The metadata holds environment constants. From gymnasium, we inherit the "render_modes",
    metadata which specifies which modes can be put into the render() method.
    At least human mode should be supported.
    The "name" metadata allows the environment to be pretty printed.
    """

    metadata = {"render_modes": ["human"], "name": "rps_v2"}

    def __init__(self, render_mode=None):
        """
        The init method takes in environment arguments and
         should define the following attributes:
        - possible_agents
        - render_mode

        Note: as of v1.18.1, the action_spaces and observation_spaces attributes are deprecated.
        Spaces should be defined in the action_space() and observation_space() methods.
        If these methods are not overridden, spaces will be inferred from self.observation_spaces/action_spaces, raising a warning.

        These attributes should not be changed after initialization.
        """
        self.possible_crops = ["Rice", "Maize", "Cotton"]
        self.crop_market_prices_per_kg = [10, 20, 30]
        self.carbon_market_price_per_credit = 100
        self.possible_inputs = {"Fertilizer": 20, "Manure": 10}
        self.previous_farmer_enrollment = None
        self.previous_soc_change = None
        self.current_period = 0
        self.previous_carbon_credits_generated = None

        self.possible_upfront_payments = None
        self.market_price_per_credit = None
        self.monitoring_levels = ["low", "medium", "high"]

        self.possible_agents = ["farmer_" + str(r) for r in range(2)] + ["aggregator"]
        self.num_periods = None

        # optional: we can define the observation and action spaces here as attributes to be used in their corresponding methods
        farmer_action_space = Dict({
            "crop_choice": Discrete(len(self.possible_crops)),
            "input_choice": Discrete(len(self.possible_inputs)),
            "subscribe": Discrete(2),
        })

        aggregator_action_space = Dict({
            "offered_price_per_carbon_credit": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "monitoring_level": Discrete(len(self.monitoring_levels)),
            "upfront_payment": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "revenue_sharing": Box(low=0.0, high=100, shape=(1,), dtype=np.float32),
        })

        farmer_observation_space = Dict({
            "crop_prices": Discrete(len(self.crop_market_prices_per_kg)),
            "input_prices": Discrete(len(self.possible_inputs)),
            "offered_price_per_carbon_credit": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "upfront_payment": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "monitoring_level": Discrete(len(self.monitoring_levels)),
            "revenue_sharing": Box(low=0.0, high=100, shape=(1,), dtype=np.float32),
        })

        aggregator_observation_space = Dict({
            "carbon_market_price_per_credit": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "previous_enrollment": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "previous_carbon_credits_achieved": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "Period": Discrete(self.num_periods),
        })

        self._action_spaces = {
            "farmer_1": farmer_action_space,
            "farmer_2": farmer_action_space,
            "aggregator": aggregator_action_space,
        }
        self._observation_spaces = {
            "farmer_1": farmer_observation_space,
            "farmer_2": farmer_observation_space,
            "aggregator": aggregator_observation_space,
        }
        self.render_mode = render_mode

    # Observation space should be defined here.
    # lru_cache allows observation and action spaces to be memoized, reducing clock cycles required to get each agent's space.
    # If your spaces change over time, remove this line (disable caching).
    # @functools.lru_cache(maxsize=None)
    def observation_space(self, agent):
        # gymnasium spaces are defined and documented here: https://gymnasium.farama.org/api/spaces/
        return self._observation_spaces[agent]

    # Action space should be defined here.
    # If your spaces change over time, remove this line (disable caching).
    # @functools.lru_cache(maxsize=None)
    def action_space(self, agent):
        # We can seed the action space to make the environment deterministic.
        return self.action_spaces[agent]

    def render(self):
        """
        Renders the environment. In human mode, it can print to terminal, open
        up a graphical window, or open up some other display that a human can see and understand.
        """
        if self.render_mode is None:
            gymnasium.logger.warn(
                "You are calling render method without specifying any render mode."
            )
            return

        if len(self.agents) == 2:
            string = "Current state: Agent1: {} , Agent2: {}".format(
                MOVES[self.state[self.agents[0]]], MOVES[self.state[self.agents[1]]]
            )
        else:
            string = "Game over"
        print(string)

    def observe(self, agent):
        """
        Observe should return the observation of the specified agent. This function
        should return a sane observation (though not necessarily the most up to date possible)
        at any time after reset() is called.
        """
        # observation of one agent is the previous state of the other
        return np.array(self.observations[agent])

    def close(self):
        """
        Close should release any graphical displays, subprocesses, network connections
        or any other environment data which should not be kept around after the
        user is no longer using the environment.
        """
        pass

    def reset(self, seed=None, options=None):
        """
        Reset needs to initialize the following attributes
        - agents
        - rewards
        - _cumulative_rewards
        - terminations
        - truncations
        - infos
        - agent_selection
        And must set up the environment so that render(), step(), and observe()
        can be called without issues.
        Here it sets up the state dictionary which is used by step() and the observations dictionary which is used by step() and observe()
        """
        # Unlike gymnasium's Env, the environment is responsible for setting the random seed explicitly.
        if seed is not None:
            self.np_random, self.np_random_seed = seeding.np_random(seed)
        self.agents = self.possible_agents[:]
        self.rewards = {agent: 0 for agent in self.agents}
        self._cumulative_rewards = {agent: 0 for agent in self.agents}
        self.terminations = {agent: False for agent in self.agents}
        self.truncations = {agent: False for agent in self.agents}
        self.infos = {agent: {} for agent in self.agents}
        self.state = {agent: NONE for agent in self.agents}
        self.observations = {agent: NONE for agent in self.agents}
        self.num_moves = 0
        """
        Our AgentSelector utility allows easy cyclic stepping through the agents list.
        """
        self._agent_selector = AgentSelector(self.agents)
        self.agent_selection = self._agent_selector.next()

    def step(self, action):
        """
        step(action) takes in an action for the current agent (specified by
        agent_selection) and needs to update
        - rewards
        - _cumulative_rewards (accumulating the rewards)
        - terminations
        - truncations
        - infos
        - agent_selection (to the next agent)
        And any internal state used by observe() or render()
        """
        if (
            self.terminations[self.agent_selection]
            or self.truncations[self.agent_selection]
        ):
            # handles stepping an agent which is already dead
            # accepts a None action for the one agent, and moves the agent_selection to
            # the next dead agent,  or if there are no more dead agents, to the next live agent
            self._was_dead_step(action)
            return

        agent = self.agent_selection

        # the agent which stepped last had its _cumulative_rewards accounted for
        # (because it was returned by last()), so the _cumulative_rewards for this
        # agent should start again at 0
        self._cumulative_rewards[agent] = 0

        # stores action of current agent
        self.state[self.agent_selection] = action

        # collect reward if it is the last agent to act
        if self._agent_selector.is_last():
            # rewards for all agents are placed in the .rewards dictionary
            self.rewards[self.agents[0]], self.rewards[self.agents[1]] = REWARD_MAP[
                (self.state[self.agents[0]], self.state[self.agents[1]])
            ]

            self.num_moves += 1
            # The truncations dictionary must be updated for all players.
            self.truncations = {
                agent: self.num_moves >= NUM_ITERS for agent in self.agents
            }

            # observe the current state
            for i in self.agents:
                self.observations[i] = self.state[
                    self.agents[1 - self.agent_name_mapping[i]]
                ]
        else:
            # necessary so that observe() returns a reasonable observation at all times.
            self.state[self.agents[1 - self.agent_name_mapping[agent]]] = NONE
            # no rewards are allocated until both players give an action
            self._clear_rewards()

        # selects the next agent.
        self.agent_selection = self._agent_selector.next()
        # Adds .rewards to ._cumulative_rewards
        self._accumulate_rewards()

        if self.render_mode == "human":
            self.render()
